# Capstone — the 2026 portfolio review

**The question (one sentence, yours):** …
**As of:** 2026-10-01 · **Data:** `data/raw/nur.db` (synthetic, Nur Finance) · **Author:** …

Restart & Run All must work on a fresh copy. Every number on a slide comes from a cell of this notebook.

In [1]:
from pathlib import Path
import hashlib
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")                      # this notebook lives in data/python/week8/, so "../.." is the data/ folder
con = sqlite3.connect(DATA / "raw" / "nur.db")
ASOF = "2026-10-01"
sql = lambda q: pd.read_sql(q, con)       # sql("SELECT ...") returns a DataFrame
print(sql("SELECT COUNT(*) AS loans FROM loans"))


def check(value, fingerprints, dp=0):
    """Right or wrong without showing the answer: the rounded value's fingerprint must be one of the expected ones."""
    fp = hashlib.sha256(f"nur-w8:{round(float(value), dp):.{dp}f}".encode()).hexdigest()[:12]
    assert fp in fingerprints, "not yet: this result is not the expected one — check your function and run again"

   loans
0   7828


In [2]:
# The course's clean layer (week 3 Day 4), created as TEMP views: they live only in this connection, nur.db is not changed.
con.executescript("""
CREATE TEMP VIEW IF NOT EXISTS v_payments_clean AS
WITH ranked AS (
  SELECT payment_id, loan_id, installment_id, paid_at, amount,
         REPLACE(LOWER(TRIM(method)), 'bank transfer', 'transfer') AS method,
         ROW_NUMBER() OVER (PARTITION BY installment_id, paid_at, amount ORDER BY payment_id) AS rn
  FROM payments)
SELECT payment_id, loan_id, installment_id, paid_at, amount, method FROM ranked WHERE rn = 1;
CREATE TEMP VIEW IF NOT EXISTS v_loan_status AS
WITH unpaid AS (
  SELECT i.loan_id, i.due_date FROM installments i
  LEFT JOIN v_payments_clean p ON p.installment_id = i.installment_id AND p.paid_at <= '2026-10-01'
  WHERE i.due_date <= '2026-10-01'
  GROUP BY i.installment_id, i.loan_id, i.due_date, i.total_due
  HAVING COALESCE(SUM(p.amount), 0) < i.total_due),
dpd AS (SELECT loan_id, CAST(julianday('2026-10-01') - julianday(MIN(due_date)) AS INTEGER) AS dpd FROM unpaid GROUP BY loan_id),
outst AS (SELECT loan_id, SUM(principal_due) AS outstanding FROM installments WHERE due_date > '2026-10-01' GROUP BY loan_id)
SELECT l.loan_id, l.customer_id, p.code AS product, p.channel, b.region, l.risk_grade, l.status, l.disbursed_at, l.principal,
       l.term_months, COALESCE(o.outstanding, 0) AS outstanding, COALESCE(d.dpd, 0) AS dpd,
       CASE WHEN COALESCE(d.dpd, 0) = 0 THEN '0' WHEN d.dpd <= 30 THEN '1-30' WHEN d.dpd <= 60 THEN '31-60'
            WHEN d.dpd <= 90 THEN '61-90' ELSE '90+' END AS bucket
FROM loans l JOIN products p ON p.product_id = l.product_id JOIN branches b ON b.branch_id = l.branch_id
LEFT JOIN dpd d ON d.loan_id = l.loan_id LEFT JOIN outst o ON o.loan_id = l.loan_id;
""")
print(sql("SELECT COUNT(*) AS loans, SUM(outstanding) AS outstanding FROM v_loan_status"))

   loans   outstanding
0   7828  208723504430


## 0. Definitions (one line each)

| Metric | Definition |
|---|---|
| Outstanding | … |
| PAR30 / NPL90 | … |
| FPD30 | … |
| Approval rate | … |
| Collection rate | … |
| Active customer | … |
| Funnel step | … |
| Repeat borrowing | … |
| Net contribution per loan | … |

## 1. Data checks — every known problem, its handling and the rows affected

In [3]:
# your code

## 2. Sanity numbers — reproduce the fifteen numbers of Day 3, section 6

In [4]:
# your code

## Q1. Growth

Disbursements and outstanding, Jan–Sep 2026 vs Jan–Sep 2025, by product, channel, region; mix; ticket.

In [5]:
# your code

**Finding (a candidate slide title, one sentence with a number):** …

## Q2. Risk

FPD30 by channel, product, grade and period; vintage curves by channel and grade; PAR30/NPL by product and region; a roll-rate matrix.

In [6]:
# your code

**Finding (a candidate slide title, one sentence with a number):** …

## Q3. Collections efficiency

Delinquent loans and balance by bucket; share contacted in September; actions by bucket; recorded outcome vs actual payment.

In [7]:
# your code

**Finding (a candidate slide title, one sentence with a number):** …

## Q4. The app funnel

Day 1's funnel (app sessions only, ending at money), by device and version; repeat borrowing by cohort.

In [8]:
# your code

**Finding (a candidate slide title, one sentence with a number):** …

## Q5. Unit economics

Day 2's table with the assumptions printed; one sensitivity that matters for a recommendation.

In [9]:
# your code

**Finding (a candidate slide title, one sentence with a number):** …

## Q6. Deep-dive

Your question: …

In [10]:
# your code

**Finding (a candidate slide title, one sentence with a number):** …

## Recommendations

| # | Recommendation | Evidence (section) | Owner | By when | Worth / protects |
|---|---|---|---|---|---|
| 1 | … | … | … | … | … |
| 2 | … | … | … | … | … |
| 3 | … | … | … | … | … |

**Assumptions and risks:** …